# Bull BBR Reversal - Stats

## Import Libs

In [2]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Get Price Data

In [597]:
PATH = os.getcwd()
FILE = "../output/FE_V2_GBPUSD_15mins_1yr_End_20260311.csv"
df = pd.read_csv(FILE)
df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
df.set_index("Date", inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 24580 entries, 2025-03-11 17:15:00-04:00 to 2026-03-11 16:45:00-04:00
Data columns (total 49 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Open           24580 non-null  float64
 1   High           24580 non-null  float64
 2   Low            24580 non-null  float64
 3   Close          24580 non-null  float64
 4   Idx            24580 non-null  int64  
 5   Body           24580 non-null  float64
 6   Range          24580 non-null  float64
 7   UWick          24580 non-null  float64
 8   LWick          24580 non-null  float64
 9   Close_%High    24580 non-null  float64
 10  Open_%High     24580 non-null  float64
 11  Iday_Idx       24580 non-null  int64  
 12  Iday_High      24580 non-null  float64
 13  Iday_Low       24580 non-null  float64
 14  Iday_Range     24580 non-null  float64
 15  Close_%DHigh   24580 non-null  float64
 16  Open_%DHigh    24580 non-null  float64
 17  Yda

## Low vs ADR% Stop Loss - 38.2% ADR Target

In [598]:
def get_bull_bbr_pip_gain(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series  
        ):
    """Get the pip gain and apply to df"""

    low_win = 0
    low_loss = 0
    adr_win = 0
    adr_loss = 0
    low_target_pips = 0 
    adr_target_pips = 0 
    low_sl_pips = 0
    adr_sl_pips = 0
    low_gain = 0
    adr_gain = 0
    low_trade_start =  None
    low_trade_end = None
    adr_trade_start =  None
    adr_trade_end = None
    
    if df["Bull_BBR"] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        low_stop = False 
        low_sl_ts = None # get timestamp when stopped out
        adr_stop = False # update if stopped 
        adr_sl_ts = None # get timestamp when stopped out
        stoploss_low = max(low.iloc[idx-1], low.iloc[idx])

        for i in range(len(sl_window)):
            if sl_window.iloc[i] <= stoploss_low : #stoploss_high:
                low_stop = True # trade hit stop loss
                # get stop loss timestamp
                low_sl_ts = sl_window.iloc[i:i+1].index[0]
                break 
        for i in range(len(sl_window)): 
            if sl_window.iloc[i] <= (df["Close"] - df["ADR"] * 0.25 ): #(stoploss_high + df["ATR"]):
                adr_stop = True # trade hit stop loss
                # get stop loss timestamp
                adr_sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if low_stop is False:
            if close_window.empty is False:
                low_sl_ts = close_window.iloc[-1:].index[0]
            else:
                low_sl_ts = START+TD
        if adr_stop is False:
            if close_window.empty is False:
                adr_sl_ts = close_window.iloc[-1:].index[0]
            else:
                adr_sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df["ADR"] * 0.382)

        # High trade window
        low_tp_window = high[START+TD:low_sl_ts+TD]
        low_trade_start = START+TD
        low_trade_end = low_sl_ts
        low_target_pips = tp - df["Close"]
        low_sl_pips = stoploss_low - df["Close"]
        if low_tp_window.max() >= tp:
            low_win = 1
            low_gain = low_target_pips
        else:
            low_loss = 1
            if low_stop is True:
                low_gain = low_sl_pips
            else:
                low_gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0           
        # ATR trade window
        adr_tp_window = high[START+TD:adr_sl_ts+TD]
        adr_trade_start = START+TD
        adr_trade_end = adr_sl_ts
        adr_target_pips = tp - df["Close"]
        adr_sl_pips = (df["Close"] - df["ADR"] * 0.25) - df["Close"] 
        if adr_tp_window.max() >= tp:
            adr_win = 1
            adr_gain = tp - df["Close"]
        else:
            adr_loss = 1
            if adr_stop is True:
                adr_gain = adr_sl_pips
            else:
                adr_gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
    
    data = low_win, low_loss, \
        adr_win, adr_loss, \
        low_target_pips, adr_target_pips, \
        low_sl_pips, adr_sl_pips, \
        low_gain, adr_gain, \
        adr_trade_start, adr_trade_end, \
        low_trade_start, low_trade_end
    
    return data

# Set new columns 
pct_50_idr_cols = [
    "Low_Win", "Low_Loss", 
    "ADR_Win", "ADR_Loss", 
    "Low_TP", "ADR_TP", 
    "Low_SL", "ADR_SL", 
    "Low_Gain", "ADR_Gain",
    "ADR_Trade_Start", "ADR_Trade_End",
    "Low_Trade_Start", "Low_Trade_End"
    ]

df[pct_50_idr_cols] = df.apply(get_bull_bbr_pip_gain, axis=1, args=[df["High"], df["Low"], df["Close"]], result_type='expand')


In [599]:
df2 = df.query("Bull_BBR == True ").between_time("01:00", "16:00")

In [600]:
# Stats - High as SL
low_win_count = df2.query("Bull_BBR == True and Low_Win > 0")["Bull_BBR"].count()
low_loss_count = df2.query("Bull_BBR == True and Low_Loss > 0")["Bull_BBR"].count()
low_total_trades = low_win_count + low_loss_count
low_win_rate = low_win_count/low_total_trades * 100
low_win = df2.query("Bull_BBR == True and Low_Gain > 0")["Low_Gain"]
low_loss = df2.query("Bull_BBR == True and Low_Gain < 0")["Low_Gain"]
low_win_avg_pips = low_win.mean()
low_loss_avg_pips = low_loss.mean()
low_win_pips = low_win.sum()
low_loss_pips = low_loss.sum()
low_total_pips = low_win_pips + low_loss_pips

low_pct_50_idr = {
    "Win_Count": low_win_count,
    "Loss_Count": low_loss_count,
    "Total_Trades": low_total_trades,
    "Win_Rate": low_win_rate,
    "Avg_Win": low_win_avg_pips,
    "Avg_Loss": low_loss_avg_pips,
    "Win_Pips": low_win_pips,
    "Loss_Pips": low_loss_pips,
    "Total_Pips": low_total_pips
}

# Stats - High + ATR as SL
adr_win_count = df2.query("Bull_BBR == True and ADR_Win > 0")["Bull_BBR"].count()
adr_loss_count = df2.query("Bull_BBR == True and ADR_Loss > 0")["Bull_BBR"].count()
adr_total_trades = adr_win_count + adr_loss_count
adr_win_rate = adr_win_count/adr_total_trades * 100
adr_win = df2.query("Bull_BBR == True and ADR_Gain > 0")["ADR_Gain"]
adr_loss = df2.query("Bull_BBR == True and ADR_Gain < 0")["ADR_Gain"]
adr_win_avg_pips = adr_win.mean()
adr_loss_avg_pips = adr_loss.mean()
adr_win_pips = adr_win.sum()
adr_loss_pips = adr_loss.sum()
adr_total_pips = adr_win_pips + adr_loss_pips

adr_pct_50_idr = {
    "Win_Count": adr_win_count,
    "Loss_Count": adr_loss_count,
    "Total_Trades": adr_total_trades,
    "Win_Rate": adr_win_rate,
    "Avg_Win": adr_win_avg_pips,
    "Avg_Loss": adr_loss_avg_pips,
    "Win_Pips": adr_win_pips,
    "Loss_Pips": adr_loss_pips,
    "Total_Pips": adr_total_pips
}

pd.DataFrame([low_pct_50_idr, adr_pct_50_idr])


,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
0,44,239,283,15.547703,0.002665,-0.000741,0.162555,-0.164450,-0.001895
1,81,202,283,28.621908,0.002461,-0.001990,0.312508,-0.310438,0.002069


In [595]:
df2[["RSI", "Iday_Range", "RSI_DVG", "SMA32_Slope", "SMA16_Slope",*pct_50_idr_cols]].iloc[200:].query("ADR_Gain > 0")

,RSI,Iday_Range,RSI_DVG,SMA32_Slope,SMA16_Slope,Low_Win,Low_Loss,ADR_Win,ADR_Loss,Low_TP,ADR_TP,Low_SL,ADR_SL,Low_Gain,ADR_Gain,ADR_Trade_Start,ADR_Trade_End,Low_Trade_Start,Low_Trade_End
Date,,,,,,,,,,,,,,,,,,,
2024-11-25 03:45:00-05:00,35.774245,0.005865,NaN,-37.905960,-58.094924,1.0,0.0,1.0,0.0,0.003439,0.003439,-0.001190,-0.002251,0.003439,0.003439,2024-11-25 04:00:00-05:00,2024-11-25 16:45:00-05:00,2024-11-25 04:00:00-05:00,2024-11-25 10:45:00-05:00
2024-11-26 03:15:00-05:00,51.877864,0.006640,NaN,22.772258,-27.465035,1.0,0.0,1.0,0.0,0.003477,0.003477,-0.001050,-0.002276,0.003477,0.003477,2024-11-26 03:30:00-05:00,2024-11-26 13:30:00-05:00,2024-11-26 03:30:00-05:00,2024-11-26 11:45:00-05:00
2024-11-27 01:00:00-05:00,50.791761,0.001365,NaN,0.477454,-11.653841,1.0,0.0,1.0,0.0,0.003531,0.003531,-0.000295,-0.002311,0.003531,0.003531,2024-11-27 01:15:00-05:00,2024-11-27 16:45:00-05:00,2024-11-27 01:15:00-05:00,2024-11-27 16:45:00-05:00
2024-11-28 02:45:00-05:00,38.249735,0.003745,NaN,-29.809114,-31.138972,0.0,1.0,1.0,0.0,0.003566,0.003566,-0.000475,-0.002334,-0.000475,0.003566,2024-11-28 03:00:00-05:00,2024-11-28 16:45:00-05:00,2024-11-28 03:00:00-05:00,2024-11-28 03:00:00-05:00
2024-11-29 09:15:00-05:00,40.268276,0.007825,NaN,-35.291337,-31.182676,1.0,0.0,1.0,0.0,0.003561,0.003561,-0.000570,-0.002331,0.003561,0.003561,2024-11-29 09:30:00-05:00,2024-11-29 16:45:00-05:00,2024-11-29 09:30:00-05:00,2024-11-29 16:45:00-05:00
2024-12-03 07:30:00-05:00,39.793070,0.005855,NaN,7.447992,-57.328655,0.0,1.0,1.0,0.0,0.003629,0.003629,-0.000370,-0.002375,-0.000370,0.003629,2024-12-03 07:45:00-05:00,2024-12-03 16:45:00-05:00,2024-12-03 07:45:00-05:00,2024-12-03 07:45:00-05:00
2024-12-03 10:30:00-05:00,46.448184,0.006035,NaN,-20.188896,-36.408775,1.0,0.0,1.0,0.0,0.003629,0.003629,-0.001705,-0.002375,0.003629,0.003629,2024-12-03 10:45:00-05:00,2024-12-03 16:45:00-05:00,2024-12-03 10:45:00-05:00,2024-12-03 16:45:00-05:00
2024-12-09 07:45:00-05:00,49.215224,0.006650,NaN,38.440789,-17.787973,1.0,0.0,1.0,0.0,0.003683,0.003683,-0.000565,-0.002411,0.003683,0.003683,2024-12-09 08:00:00-05:00,2024-12-09 16:45:00-05:00,2024-12-09 08:00:00-05:00,2024-12-09 14:45:00-05:00
2024-12-11 14:00:00-05:00,44.583561,0.006855,NaN,17.354025,-44.028978,0.0,1.0,0.0,1.0,0.003702,0.003702,-0.000495,-0.002423,0.001155,0.001155,2024-12-11 14:15:00-05:00,2024-12-11 16:45:00-05:00,2024-12-11 14:15:00-05:00,2024-12-11 16:45:00-05:00
